# Clase 15 (Cierre) — Proyecto integrador: Extract → Transform → Load

## Objetivos de la clase

Al terminar esta clase vas a poder:

- Explicar por qué ningún proyecto real de datos usa una sola herramienta, sino una **secuencia**: extracción, transformación y carga (ETL).
- Reconocer, en un problema real, qué capítulo del curso aporta cada pieza de la solución.
- Reproducir de punta a punta el proyecto "Data Engineer junior": extraer una tabla de PIB por país, limpiarla y transformarla, y exportarla a CSV.
- Construir código de scraping **robusto**, con un plan B cuando la fuente externa no responde.

Esta es la última clase del curso. No vas a aprender sintaxis nueva: vas a ver cómo todo lo que ya sabes (Bloque 1: Python base; Bloque 2: archivos, NumPy, Pandas; Bloque 3: APIs y scraping) encaja en un solo flujo de trabajo.

## De la teoría a la práctica

Repasa por un segundo todo lo que ya sabes hacer:

- **Bloque 1** (Clases 1-8): variables, funciones, tus propias clases (POO), manejo de errores.
- **Bloque 2** (Clases 9-11): archivos, cálculo numérico con NumPy, limpieza y exploración de tablas con Pandas.
- **Bloque 3** (Clases 12-14): pedir datos a una API, extraer datos directamente de una página web (scraping).

Cada clase se sintió como una pieza aislada mientras la cursabas. Hoy vas a ver que esas piezas encajan: un problema real casi nunca se resuelve con una sola herramienta, sino encadenando varias siguiendo el ciclo **Extract → Transform → Load (ETL)**.

## El encargo

Una empresa internacional que quiere expandirse a distintos países te contrata como Data Engineer junior. Tu tarea: armar un script que extraiga la lista de las **10 economías más grandes del mundo**, ordenadas de mayor a menor PIB en miles de millones de dólares (billones USD, redondeado a 2 decimales), según los datos que publica el Fondo Monetario Internacional (FMI).

Los datos están en una tabla dentro de una página de Wikipedia archivada por la Wayback Machine (para que el resultado sea reproducible):

```
https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29
```

Este es el trabajo real de un Data Engineer: la fuente casi nunca viene en el formato que necesitas, y ni siquiera está garantizado que la fuente externa responda siempre. Por eso, en el código de esta clase vas a ver un `try/except`: **intentamos el scraping real** contra esa URL y, si falla o tarda demasiado (una red inestable, la Wayback Machine caída, un firewall del aula), seguimos con una **tabla de respaldo** ficticia pero realista, para que la clase nunca se frene por un problema de conectividad. Todo el resto del flujo (limpieza, transformación, exportación) es idéntico se use la tabla real o la de respaldo.

## Paso 0 — Setup: importar librerías

Como en cualquier notebook de Pandas/NumPy (Clases 10 y 11), importamos todo al principio.

In [ ]:
import numpy as np
import pandas as pd

# Suprimimos warnings para que la salida de la clase quede prolija
import warnings
warnings.filterwarnings('ignore')

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## Paso 1 — Extraer (Extract): de una página web a un DataFrame

*Herramienta: Clase 14, Web Scraping.*

En el capítulo de Web Scraping viste `BeautifulSoup` buscando etiquetas a mano. Pandas te resuelve el trabajo pesado con una sola función: `pd.read_html()` recorre el HTML de una URL, encuentra **todas** las tablas de la página y te las devuelve ya convertidas en una lista de DataFrames.

Fíjate: `pd.read_html()` **es** scraping. La diferencia con lo que viste en la clase anterior es que Pandas te ahorra escribir el recorrido del árbol HTML porque el patrón "tabla de una página web → tabla de datos" es tan común que ya viene resuelto.

**Nota sobre robustez:** el scraping en vivo depende de que la Wayback Machine esté disponible y responda rápido en este momento. Para que la clase no se detenga por eso, probamos el scraping real con un `try/except` y, si falla, seguimos con una tabla de respaldo. **Si tienes una conexión estable y quieres reproducir el proyecto original tal cual**, puedes ejecutar únicamente el bloque del `try` contra la URL real indicada arriba.

In [ ]:
URL = "https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"

df = None

try:
    # --- Intento real de scraping (puede tardar o fallar si no hay internet estable) ---
    # pd.read_html() devuelve una LISTA de DataFrames, uno por cada tabla que encuentra en la página
    tables = pd.read_html(URL)

    # La tabla que necesitamos (país + PIB del FMI) es la tercera de la página
    df = tables[3]
    fuente = "scraping real (Wayback Machine)"

except Exception as error:
    # --- Plan B: si el scraping falla (sin internet, timeout, la página cambió), usamos
    # una tabla de respaldo ficticia pero realista, con la MISMA estructura que la tabla
    # original: nombre de país en la columna 0 y PIB del FMI en millones USD en la columna 2.
    print(f"No se pudo hacer scraping en vivo ({type(error).__name__}: {error}).")
    print("Seguimos con una tabla de respaldo para no frenar la clase.\n")

    datos_respaldo = {
        0: ['World', 'United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom'],
        1: ['-', '-', '-', '-', '-', '-', '-'],  # columna que no usamos (ej. región/año)
        2: [105000000, 26950000, 17700000, 4430000, 4230000, 3730000, 3330000],  # PIB (FMI) en millones USD
    }
    df = pd.DataFrame(datos_respaldo)
    fuente = "tabla de respaldo (offline)"

print("Fuente de datos usada:", fuente)
df.head()

## Paso 2 — Transformar, parte 1: dar forma a la tabla

*Herramienta: Clase 11, Pandas (`.iloc`).*

La tabla real de Wikipedia trae columnas con nombres compuestos y encabezados agrupados. En vez de pelear con esos nombres, reemplazamos los encabezados por su posición numérica: así podemos referirnos a cualquier columna por su índice, sin importar cómo se llamaba originalmente. Esto funciona igual sobre la tabla real o sobre la de respaldo, porque ambas tienen la misma estructura de columnas.

In [ ]:
# Reemplazamos los encabezados de texto por números de columna (0, 1, 2, ...)
df.columns = range(df.shape[1])

# Nos quedamos solo con la columna 0 (nombre del país) y la columna 2 (PIB según el FMI)
df = df[[0, 2]]

# Con .iloc (Clase 11) seleccionamos las filas 1 a 10: el top 10 de economías del mundo
# (la fila 0 es "World"/total, no nos interesa para el ranking)
df = df.iloc[1:11, :]

# Renombramos las columnas para que el resultado sea legible
df.columns = ['Country', 'GDP (Million USD)']

df

## Paso 3 — Transformar, parte 2: convertir unidades con NumPy

*Herramienta: Clase 10, NumPy + Clase 11, Pandas.*

El FMI reporta el PIB en millones de dólares, pero el encargo pide miles de millones (billones USD). Antes de poder dividir necesitamos asegurarnos de que la columna sea numérica — al venir de una tabla HTML (o de un diccionario con enteros, en nuestro caso de respaldo), conviene forzar el tipo con `.astype()` para que el código funcione igual en ambos casos.

In [ ]:
# Nos aseguramos de que la columna sea numérica antes de operar con ella
df['GDP (Million USD)'] = df['GDP (Million USD)'].astype(int)

# Operación aritmética sobre la columna completa: de millones a miles de millones.
# Esto es vectorizado (sin bucle for) porque una columna de Pandas se apoya en NumPy.
df[['GDP (Million USD)']] = df[['GDP (Million USD)']] / 1000

# np.round() (Clase 10) redondea cada valor a 2 decimales, tal como pidió el encargo
df[['GDP (Million USD)']] = np.round(df[['GDP (Million USD)']], 2)

# Ahora que la unidad cambió, el nombre de la columna también tiene que reflejarlo
df = df.rename(columns={'GDP (Million USD)': 'GDP (Billion USD)'})

df

## Paso 4 — Cargar (Load): dejar el resultado en un CSV

*Herramienta: Clase 9, Archivos.*

El último paso del ETL es dejar el resultado en algún lugar donde otra persona (o tú mismo, la semana que viene) pueda volver a usarlo sin repetir todo el trabajo de scraping y limpieza. Con lo que ya sabes de la clase de archivos, esto es una sola línea.

In [ ]:
df.to_csv('./Largest_economies.csv')
print("Archivo 'Largest_economies.csv' generado correctamente.")

# Verificamos releyendo el archivo
pd.read_csv('./Largest_economies.csv', index_col=0)

## El flujo completo, de punta a punta

Así se ve el proyecto completo si lo escribimos como un único script, encadenando los cuatro pasos (con la misma lógica de `try/except` para que sea robusto en cualquier entorno):

```python
import numpy as np
import pandas as pd

# --- EXTRACT (Clase 14: Web Scraping) ---
URL = "https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"
try:
    tables = pd.read_html(URL)
    df = tables[3]
except Exception:
    datos_respaldo = {
        0: ['World', 'United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom'],
        1: ['-', '-', '-', '-', '-', '-', '-'],
        2: [105000000, 26950000, 17700000, 4430000, 4230000, 3730000, 3330000],
    }
    df = pd.DataFrame(datos_respaldo)

# --- TRANSFORM, parte 1: dar forma a la tabla (Clase 11: Pandas) ---
df.columns = range(df.shape[1])
df = df[[0, 2]]
df = df.iloc[1:11, :]
df.columns = ['Country', 'GDP (Million USD)']

# --- TRANSFORM, parte 2: convertir unidades (Clase 10: NumPy) ---
df['GDP (Million USD)'] = df['GDP (Million USD)'].astype(int)
df[['GDP (Million USD)']] = df[['GDP (Million USD)']] / 1000
df[['GDP (Million USD)']] = np.round(df[['GDP (Million USD)']], 2)
df = df.rename(columns={'GDP (Million USD)': 'GDP (Billion USD)'})

# --- LOAD (Clase 9: Archivos) ---
df.to_csv('./Largest_economies.csv')
```

Diez líneas de lógica real, apoyadas en cuatro clases distintas del curso. Ese es el patrón que te vas a encontrar una y otra vez en cualquier trabajo con datos: ningún paso es difícil por separado, la habilidad está en saber en qué orden encadenarlos y qué herramienta usar en cada uno.

## Otros proyectos del curso que ya seguían este mismo patrón

Sin llamarlo "ETL" en su momento, ya resolviste el mismo ciclo varias veces:

- **Estadísticas de la NBA vía API** (Clase 13): pedir a la API (`extract`) → tabular en un DataFrame y filtrar por equipo (`transform`) → comparar promedios y graficar.
- **TopSellingAlbums** (Clase 11): cargar un CSV (`extract`) → seleccionar columnas con `df['Length']` vs. `df[['Length']]` y filas con `.iloc`/`.loc` (`transform`) → el mismo par de herramientas que usaste para recortar la tabla del PIB.
- **Dataset de diabetes** (Clase 11): explorar con `.head()`, `.shape()`, `.info()`, `.describe()` antes de sacar conclusiones — el primer paso de cualquier `transform` sobre un dataset nuevo.
- **TextAnalyzer** (Clase 7, POO): una clase que organiza limpieza de texto y conteo de frecuencias — la misma idea de encadenar pasos, aplicada con tus propias clases en vez de con Pandas.

## Resumen de la clase

- Ningún proyecto real usa una sola herramienta: casi siempre vas a combinar extracción (scraping o API), transformación (Pandas y NumPy) y carga (archivos) en la misma secuencia.
- El ciclo **Extract → Transform → Load** es literalmente el orden en que resolviste el proyecto del PIB, paso por paso.
- `pd.read_html()` es tu primera opción cuando los datos que necesitas ya están en una tabla HTML.
- `.iloc` (y `.loc`) son las herramientas que usas constantemente para recortar exactamente las filas y columnas que necesitas.
- Las operaciones aritméticas sobre columnas completas de un DataFrame (como convertir millones a miles de millones) funcionan porque Pandas se apoya en el comportamiento vectorizado de NumPy.
- Un scraper "de producción" nunca asume que la fuente externa siempre va a responder: el patrón `try/except` con un plan B (o al menos un buen mensaje de error) es parte del trabajo, no un extra.
- Con esto se cierra el curso: la siguiente habilidad ya no es aprender más sintaxis de Python, sino practicar el criterio de elegir qué herramienta usar en cada paso de un problema nuevo.

### Para seguir practicando por tu cuenta

Elige un dataset público que te interese de verdad y recorre el ciclo completo: **extrae** (`pd.read_csv()`, `pd.read_html()`, una API con `requests`...), **transforma** (selecciona columnas/filas, corrige tipos con `.astype()`, aplica alguna operación con Pandas o NumPy) y **carga** el resultado a un archivo. No hay solución única — el objetivo es que sientas que el flujo completo ya es tuyo.